In [1]:
# =====================================================================
# 2. REPRODUCIBILITY — set ALL random seeds
#    Run this cell BEFORE any data loading / augmentation / model init.
# =====================================================================
import os
SEED = 314

# --- 1. Python hash seed ---
# NOTE: For full effect this must be set BEFORE the kernel starts
# (e.g. launch notebook with: PYTHONHASHSEED=42 jupyter lab).
# Setting it here covers subprocesses & future kernels only.
os.environ['PYTHONHASHSEED'] = str(SEED)
# Required by deterministic CUDA ops (e.g. torch.unique, some reductions)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# torch.use_deterministic_algorithms(True)   # uncomment ONLY for bit-exact reproducibility;
#              

In [2]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
# --------------------------------------------------

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [3]:


# --- 2. Python `random` module (used by augmentation code as `pyrandom`) ---
random.seed(SEED)

# --- 3. NumPy (mixup beta sampling, TTA subset sampling, etc.) ---
np.random.seed(SEED)

# --- 4. PyTorch (CPU + all GPUs) ---
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)          # multi-GPU safety

# --- 5. cuDNN determinism (slightly slower, reproducible) ---
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

                               # may error out on some autocast/GradScaler ops

# --- 6. Hugging Face transformers seed ---
import transformers
transformers.set_seed(SEED)

# --- 7. Shared RNG generator for DataLoader shuffling ---
rng_generator = torch.Generator()
rng_generator.manual_seed(SEED)

print(f"✅ Reproducibility ON — SEED = {SEED}")
print("   python random | numpy | torch (cpu+cuda) | cuDNN deterministic | transformers | DataLoader generator")

✅ Reproducibility ON — SEED = 314
   python random | numpy | torch (cpu+cuda) | cuDNN deterministic | transformers | DataLoader generator


In [4]:
with open('MCMFPP/train.txt') as f:
    lines = f.read().split('>')
    data = []
    for line in lines:
        if line.strip() != '':
            seq = line.split('\n')[1].strip()
            labels = [int(a) for a in line.split('\n')[0].strip()]
            data.append([seq]+ labels)

train_df = pd.DataFrame(data=data, columns=['sequence', 'AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP'])
            
with open('MCMFPP/test.txt') as f:
    lines = f.read().split('>')
    data = []
    for line in lines:
        if line.strip() != '':
            seq = line.split('\n')[1].strip()
            labels = [int(a) for a in line.split('\n')[0].strip()]
            data.append([seq]+ labels)

test_df = pd.DataFrame(data=data, columns=['sequence', 'AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP'])
    

In [5]:
train_df[['AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP']].sum()

AAP        115
ABP       1735
ACP        850
ACVP        98
ADP        400
AEP         48
AFP       1079
AHIVP       82
AHP        758
AIP       1624
AMRSAP     147
APP        218
ATP        182
AVP        568
BBP         92
BIP        274
CPP        366
DPPIP      250
QSP        171
SBP         89
THP        531
dtype: int64

# creating dataset

In [6]:
import torch
import numpy as np


import torch


# Example usage:
# seq = "ALDFR" -> Dipeptides: AL, LD, DF, FR
# vector = get_dipeptide_composition(seq)


esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")





In [7]:
class PeptideDataset(Dataset):
    """Pre-tokenizes all sequences at init time for much faster training."""

    def __init__(self, dataframe, tokenizer, label_columns, max_length=128):
        self.labels = dataframe[label_columns].values
        self.max_length = max_length
        
        # Pre-tokenize ALL sequences once
        sequences = dataframe['sequence'].tolist()
        encodings = tokenizer(
            sequences,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        self.input_ids = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        
        
        print(f"  Pre-tokenized {len(sequences)} sequences")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'labels': torch.FloatTensor(self.labels[index])
        }


In [8]:
import random as pyrandom

# --- Data augmentation for single-function peptides (MCMFPP paper, Zhao et al. 2025) ---
def augment_single_function_peptides(df, label_columns, n_mask=2):
    """Augment single-function peptides by masking random amino acids with 'X'.
    For each single-function peptide, create n_mask masked copies.
    (Skipping flip/reverse since ESM2 expects natural-direction sequences.)
    """
    single_mask = df[label_columns].sum(axis=1) == 1
    single_df = df[single_mask]
    
    augmented_rows = []
    for _, row in single_df.iterrows():
        seq = row['sequence']
        labels = row[label_columns].values
        
        for _ in range(n_mask):
            if len(seq) > 1:
                pos = pyrandom.randint(0, len(seq) - 1)
                masked_seq = seq[:pos] + 'X' + seq[pos+1:]
                new_row = {'sequence': masked_seq}
                new_row.update({col: int(labels[i]) for i, col in enumerate(label_columns)})
                augmented_rows.append(new_row)
    
    if augmented_rows:
        aug_df = pd.DataFrame(augmented_rows)
        combined = pd.concat([df, aug_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"  Augmented: {len(df)} → {len(combined)} ({len(single_df)} single-func × {n_mask} masks)")
        return combined
    return df



import random as pyrandom

SIMILAR_AA = {
    'A': ['G', 'S', 'V'], 'R': ['K', 'Q'], 'N': ['D', 'Q', 'S'],
    'D': ['N', 'E'], 'C': ['S'], 'Q': ['N', 'E', 'K'],
    'E': ['D', 'Q'], 'G': ['A', 'S'], 'H': ['R', 'Y'],
    'I': ['L', 'V', 'M'], 'L': ['I', 'V', 'M'], 'K': ['R', 'Q'],
    'M': ['I', 'L', 'V'], 'F': ['Y', 'W'], 'P': ['A'],
    'S': ['A', 'T', 'N'], 'T': ['S', 'A'], 'W': ['F', 'Y'],
    'Y': ['F', 'H', 'W'], 'V': ['I', 'L', 'A'],
}

def augment_rare_classes_substitution(df, label_columns, similar_aa, 
                                       rarity_threshold=500, 
                                       max_aug_per_class=300,
                                       n_subs=1):
    """
    Conservative AA substitution ONLY for rare classes.
    - rarity_threshold: classes with fewer positives than this are 'rare'
    - max_aug_per_class: cap to avoid over-saturating the dataset
    - n_subs: number of residues to substitute per sequence (keep at 1)
    """
    class_counts = df[label_columns].sum()
    rare_cols = class_counts[class_counts < rarity_threshold].index.tolist()
    
    if not rare_cols:
        print("  No rare classes found, skipping substitution.")
        return df

    print(f"  Rare classes targeted: {rare_cols}")
    
    augmented_rows = []
    for col in rare_cols:
        # All peptides positive for this rare class
        pos_mask = df[col] == 1
        pos_df = df[pos_mask].copy()
        
        # Cap augmentation volume per class
        if len(pos_df) > max_aug_per_class:
            pos_df = pos_df.sample(n=max_aug_per_class, random_state=42)
        
        for _, row in pos_df.iterrows():
            seq = row['sequence']
            labels = row[label_columns].values
            
            if len(seq) < 3:
                continue
            
            seq_list = list(seq)
            # Pick random positions, prefer interior residues (skip first/last)
            candidates = list(range(1, len(seq) - 1))
            pyrandom.shuffle(candidates)
            
            subs_done = 0
            for pos in candidates:
                if subs_done >= n_subs:
                    break
                aa = seq_list[pos]
                if aa in similar_aa and len(similar_aa[aa]) > 0:
                    new_aa = pyrandom.choice(similar_aa[aa])
                    seq_list[pos] = new_aa
                    subs_done += 1
            
            if subs_done > 0:
                new_row = {'sequence': ''.join(seq_list)}
                new_row.update({c: int(v) for c, v in zip(label_columns, labels)})
                augmented_rows.append(new_row)
    
    if augmented_rows:
        aug_df = pd.DataFrame(augmented_rows)
        combined = pd.concat([df, aug_df], ignore_index=True)
        # Shuffle so augmented samples are distributed across epochs
        combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"  Substitution augment: {len(df)} → {len(combined)} (+{len(aug_df)} for rare classes)")
        return combined
    return df




esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
LABEL_COLUMNS = ['AAP', 'ABP', 'ACP', 'ACVP', 'ADP', 'AEP', 'AFP', 'AHIVP',
       'AHP', 'AIP', 'AMRSAP', 'APP', 'ATP', 'AVP', 'BBP','BIP', 'CPP', 'DPPIP', 'QSP', 'SBP', 'THP']

MAX_LEN = 60

# Augment training data (paper optimal: n_mask=2)
# --- IN YOUR PIPELINE (run AFTER masking augmentation) ---
train_df_aug = augment_single_function_peptides(train_df, LABEL_COLUMNS, n_mask=1)
train_df_aug = augment_rare_classes_substitution(
    train_df_aug, 
    LABEL_COLUMNS, 
    SIMILAR_AA,
    rarity_threshold=800,      # classes with <500 samples
    max_aug_per_class=300,     # don't add more than 200 per rare class
    n_subs=1                   # ONLY 1 substitution per sequence
)

train_dataset = PeptideDataset(train_df_aug, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)
test_dataset = PeptideDataset(test_df, esm_tokenizer, LABEL_COLUMNS, MAX_LEN)

  Augmented: 7899 → 14617 (6718 single-func × 1 masks)
  Rare classes targeted: ['AAP', 'ACVP', 'ADP', 'AEP', 'AHIVP', 'AMRSAP', 'APP', 'ATP', 'BBP', 'BIP', 'CPP', 'DPPIP', 'QSP', 'SBP']
  Substitution augment: 14617 → 17731 (+3114 for rare classes)
  Pre-tokenized 17731 sequences
  Pre-tokenized 1975 sequences


In [9]:
# Seeded generator → reproducible shuffle order across runs
# (rng_generator created in the Reproducibility cell above)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          generator=rng_generator)
test_loader = DataLoader(test_dataset, batch_size=32)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM

# =====================================================================
# Model v19 (Lite): Sequence-Level Cross-Attention + Task Query Decoder
# Parameter Count: ~19.6 Million
# =====================================================================

class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    """
    Mask-aware Squeeze-and-Excitation block.

    Input:
        x        : [B, C, L]
        seq_mask : [B, L], 1 for valid tokens, 0 for PAD

    The channel descriptor is computed using only valid sequence positions.
    """

    def __init__(self, channels, reduction=4):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x, seq_mask):
        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Remove PAD contributions
        x_masked = x * mask

        # Number of valid positions per sequence
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(dtype=x.dtype)

        # Masked global average pooling over sequence dimension
        # [B, C, L] -> [B, C]
        channel_descriptor = (
            x_masked.sum(dim=-1) / lengths
        )

        # Channel-wise gates
        gates = self.fc(channel_descriptor).unsqueeze(-1)

        # Apply channel recalibration
        return x * gates

class ConvBlock(nn.Module):
    """
    Mask-aware two-layer 1D convolution block.

    Uses LayerNorm instead of BatchNorm so that padded positions
    do not participate in batch/sequence normalization statistics.
    """

    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm1 = nn.LayerNorm(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm2 = nn.LayerNorm(out_channels)

    @staticmethod
    def apply_layernorm(x, norm):
        """
        Conv output: [B, C, L]
        LayerNorm expects normalized dimension at the end.
        """
        x = x.transpose(1, 2)      # [B, L, C]
        x = norm(x)
        x = x.transpose(1, 2)      # [B, C, L]
        return x

    def forward(self, x, mask):
        """
        x    : [B, C, L]
        mask : [B, 1, L]
        """
        # First convolution
        y = self.conv1(x)
        y = self.apply_layernorm(
            y,
            self.norm1
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations
        y = y * mask

        # Second convolution
        y = self.conv2(y)

        y = self.apply_layernorm(
            y,
            self.norm2
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations again
        y = y * mask

        return y

class EnhancedCNN1D(nn.Module):
    """
    Mask-aware multi-scale CNN for peptide/protein sequences.

    Branches:
        kernel 3 -> effective receptive field 5
        kernel 5 -> effective receptive field 9
        kernel 7 -> effective receptive field 13

    Each branch contains two convolutional layers.

    Output:
        [B, 2 * 3 * conv_dim]

    For conv_dim=128:
        output = [B, 768]
    """

    def __init__(
        self,
        vocab_size=33,
        embed_dim=128,
        conv_dim=128
    ):
        super().__init__()


        # Token embedding
        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=1 # ESM TOKENIZER
        )

        # Multi-scale CNN branches
        self.branches = nn.ModuleList([
            ConvBlock(
                in_channels=embed_dim,
                out_channels=conv_dim,
                kernel_size=k
            )
            for k in [3, 5, 7]
        ])

        # Residual projection
        self.res_proj = nn.Conv1d(
            embed_dim,
            conv_dim,
            kernel_size=1,
            bias=False
        )

        # Squeeze-and-Excitation
        self.se = SEBlock(
            channels=conv_dim * 3,
            reduction=4
        )

        self.dropout = nn.Dropout(0.2)

        # Three branches × conv_dim channels
        # Max pooling + mean pooling
        self.hidden_size = conv_dim * 3 * 2

    def forward(self, x, seq_mask):
        """
        Args:
            x:
                [B, L] token IDs

            seq_mask:
                [B, L]
                1 = valid token
                0 = PAD
        Returns:
            CNN feature vector:
                [B, hidden_size]

        For conv_dim=128:
            [B, 768]
        """


        # Validate mask


        if seq_mask is None:
            raise ValueError(
                "seq_mask must be provided to EnhancedCNN1D. "
                "CNN padding masking must never be bypassed."
            )
        # Embedding
        # [B, L] -> [B, L, embed_dim]
        x = self.embed(x)

        # [B, L, embed_dim] -> [B, embed_dim, L]
        x = x.transpose(1, 2)

        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Explicitly zero PAD embeddings
        x = x * mask

        # Residual pathway
        res = self.res_proj(x)

        # Remove PAD activations
        res = res * mask

        # Multi-scale branches
        outs = []

        for branch in self.branches:
            y = branch(x, mask)
            # Residual connection
            y = y + res
            # Guarantee PAD = 0 after residual addition
            y = y * mask
            outs.append(y)

        # Concatenate branches
        # [B, 128*3, L]
        combined = torch.cat(
            outs,
            dim=1
        )

        # Guarantee no PAD signal
        combined = combined * mask

        # Mask-aware SE
        combined = self.se(
            combined,
            seq_mask
        )

        # SE can theoretically produce nonzero values
        # at PAD positions, so mask once more.
        combined = combined * mask

        # MASKED GLOBAL MAX POOLING
        # PAD cannot become the maximum.
        combined_for_max = combined.masked_fill(
            seq_mask.unsqueeze(1) == 0,
            torch.finfo(combined.dtype).min
        )

        max_pooled = combined_for_max.max(
            dim=-1
        ).values


        # MASKED GLOBAL MEAN POOLING
        combined_for_mean = combined * mask

        # Actual number of valid tokens
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(
            dtype=combined.dtype
        )

        mean_pooled = (
            combined_for_mean.sum(dim=-1)
            / lengths
        )

        # FINAL REPRESENTATION
        pooled = torch.cat(
            [
                max_pooled,
                mean_pooled
            ],
            dim=1
        )

        return self.dropout(pooled)

class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)


    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768


        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6


        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        


        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        reduced_dim_size = 512
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, reduced_dim_size),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(reduced_dim_size)
        self.ln = nn.LayerNorm(reduced_dim_size)

        binary_features_dim = 64
        self.binary_features = nn.Sequential(
            nn.Linear(512, binary_features_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(reduced_dim_size+binary_features_dim, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input, seq_mask)                          

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)


        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        fusion = self.ln(reduced + self.fusion(reduced))
        
        binary_features = self.binary_features(fusion)
        
        return binary_features, torch.cat([fusion, binary_features], dim=1)

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
            
        _, final_fusion = self._extract_features(seq_input, seq_mask)
        return self._classify(final_fusion)

    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, mask_tokens=False, mask_prob=0.15):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask, mask_prob)
        return self._extract_features(seq_input, seq_mask)

    def classify_features(self, combined):
        return self._classify(combined)

In [11]:
DEVICE    = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [12]:
DEVICE

'cuda:0'

In [13]:

FUNC_INDICES = [i for i in range(21)]
index_endpoint = {0: 'AAP',
1: 'ABP',
2: 'ACP',
3: 'ACVP',
4: 'ADP',
5: 'AEP',
6: 'AFP',
7: 'AHIVP',
8: 'AHP',
9: 'AIP',
10: 'AMRSAP',
11: 'APP',
12: 'ATP',
13: 'AVP',
14: 'BBP',
15: 'BIP',
16: 'CPP',
17: 'DPPIP',
18: 'QSP',
19: 'SBP',
20: 'THP',}

In [14]:

import copy
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # Update parameters (weights & biases)
        for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
            s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)
        
        # FIX: Update only floating point buffers (skip int buffers like num_batches_tracked)
        for s_buf, m_buf in zip(self.shadow.buffers(), model.buffers()):
            if s_buf.dtype.is_floating_point:
                s_buf.data.mul_(self.decay).add_(m_buf.data, alpha=1 - self.decay)
            else:
                # For integer buffers, just copy directly (no EMA smoothing needed)
                s_buf.data.copy_(m_buf.data)

    def forward(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)

In [15]:
torch.cuda.empty_cache()

model = PeptideNetwork(num_classes=21, mask_token_id=32).to(DEVICE)
ema = EMA(model, decay=0.999)

# --- ASL: Asymmetric Loss for Multi-Label Classification ---
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_part = targets * torch.log(p.clamp(min=1e-8))
        neg_p = (1 - p).clamp(min=1e-8)
        if self.clip > 0:
            neg_p = (neg_p + self.clip).clamp(max=1)
        neg_part = (1 - targets) * torch.log(neg_p)
        pos_weight = (1 - p) ** self.gamma_pos
        neg_weight = p ** self.gamma_neg
        loss = -(pos_weight * pos_part + neg_weight * neg_part)
        return loss.mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=3, clip=0.02)

# Optimizer — 3 LR groups (Lowered LRs to prevent early overfitting)
esm_t6_ids = set(id(p) for p in model.esm_t6_a.esm_mlm.parameters())
esm_t6_b_ids = set(id(p) for p in model.esm_t6_b.esm_mlm.parameters())

esm_t6_params = [p for p in model.parameters() if id(p) in esm_t6_ids and p.requires_grad]
esm_t6_b_params = [p for p in model.parameters() if id(p) in esm_t6_b_ids and p.requires_grad]
other_params = [p for p in model.parameters() if id(p) not in esm_t6_ids and id(p) not in esm_t6_b_ids and p.requires_grad]

optimizer = optim.AdamW([
    {'params': esm_t6_params, 'lr': 1e-5},     # Reduced from 1e-5
    {'params': esm_t6_b_params, 'lr': 2e-6},   # Reduced from 2e-6
    {'params': other_params, 'lr': 1e-4},      # Reduced from 1e-4
],  weight_decay=0.05)   

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params/1e6:.1f}M | Trainable: {trainable_params/1e6:.1f}M")
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Total: 18.7M | Trainable: 13.6M


In [16]:
epochs = 40

In [17]:
# Warmup + Cosine Annealing
steps_per_epoch = len(train_loader)
warmup_epochs = 3
total_steps = epochs * steps_per_epoch

def lr_lambda(step):
    warmup_steps = warmup_epochs * steps_per_epoch

    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)

    progress = float(step - warmup_steps) / float(
        total_steps - warmup_steps
    )

    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Warmup ({warmup_epochs} ep) + CosineAnnealing, total {epochs} epochs, {total_steps} steps")

Warmup (3 ep) + CosineAnnealing, total 40 epochs, 22200 steps


In [18]:
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import json

In [19]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix, matthews_corrcoef, roc_curve, auc, f1_score
from tqdm import tqdm

# =====================================================================
# VECTORIZED METRIC FUNCTIONS (100x faster than loop-based versions)
# =====================================================================

def Aiming(y_hat, y):
    """Precision-like: avg ratio of correctly predicted labels over predicted labels."""
    intersection = np.sum(y_hat * y, axis=1)
    predicted = np.sum(y_hat, axis=1)
    # Only count samples with at least one intersection
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / predicted[mask]
    return np.mean(scores)

def Coverage(y_hat, y):
    """Recall-like: avg ratio of correctly predicted labels over true labels."""
    intersection = np.sum(y_hat * y, axis=1)
    true_count = np.sum(y, axis=1)
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / true_count[mask]
    return np.mean(scores)

def Accuracy(y_hat, y):
    """Jaccard-like: avg ratio of intersection over union per sample."""
    intersection = np.sum(y_hat * y, axis=1)
    union = np.sum(((y_hat + y) > 0).astype(float), axis=1)
    mask = intersection > 0
    scores = np.zeros(len(y_hat))
    scores[mask] = intersection[mask] / union[mask]
    return np.mean(scores)

def AbsoluteTrue(y_hat, y):
    """Exact match ratio."""
    return np.mean(np.all(y_hat == y, axis=1).astype(float))

def AbsoluteFalse(y_hat, y):
    """Hamming loss."""
    union = np.sum(((y_hat + y) > 0).astype(float), axis=1)
    intersection = np.sum(y_hat * y, axis=1)
    m = y.shape[1]
    return np.mean((union - intersection) / m)

def evaluate(score_label, y, threshold1=0.6, threshold2=0.4):
    _, m = y.shape
    if isinstance(threshold1, (int, float)):
        threshold1 = [threshold1] * m
    threshold1 = np.array(threshold1)
    
    y_hat = np.copy(score_label)
    
    # Vectorized thresholding
    max_vals = np.max(y_hat, axis=1)
    max_idxs = np.argmax(y_hat, axis=1)
    
    # Apply per-class thresholds
    y_hat_binary = (y_hat >= threshold1[np.newaxis, :]).astype(float)
    
    # For samples with no predictions above threshold, assign the max if > threshold2
    no_pred_mask = (y_hat_binary.sum(axis=1) == 0) & (max_vals > threshold2)
    y_hat_binary[no_pred_mask, max_idxs[no_pred_mask]] = 1.0
    
    aiming = Aiming(y_hat_binary, y)
    coverage = Coverage(y_hat_binary, y)
    accuracy = Accuracy(y_hat_binary, y)
    absolute_true = AbsoluteTrue(y_hat_binary, y)
    absolute_false = AbsoluteFalse(y_hat_binary, y)
    return aiming, coverage, accuracy, absolute_true, absolute_false

print("Vectorized metrics loaded.")


Vectorized metrics loaded.


In [20]:
# --- HELPER FUNCTIONS ---
def calculate_complete_metrics(y_true, y_pred_probs, thresholds, class_names):
    results = {}
    for i in range(y_true.shape[1]):
        y_pred_cls = (y_pred_probs[:, i] >= thresholds[i]).astype(int)
        mcc = matthews_corrcoef(y_true[:, i], y_pred_cls)
        acc = accuracy_score(y_true[:, i], y_pred_cls)
        prec = precision_score(y_true[:, i], y_pred_cls, zero_division=0)
        rec = recall_score(y_true[:, i], y_pred_cls, zero_division=0)
        f1 = f1_score(y_true[:, i], y_pred_cls, zero_division=0)
        try: auc_val = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
        except ValueError: auc_val = 0.5
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_cls, labels=[0,1]).ravel()
        results[class_names[i]] = {
            "mcc": float(mcc), "threshold": float(thresholds[i]),
            "accuracy": float(acc), "precision": float(prec),
            "recall": float(rec), "f1_score": float(f1), "roc_auc": float(auc_val),
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
        }
    return results

def find_optimal_thresholds(y_true, y_pred_probs):
    optimal = []
    for i in range(y_true.shape[1]):
        best_f1, best_t = -1, 0.5
        for t in np.arange(0.15, 0.85, 0.02):
            pred = (y_pred_probs[:, i] >= t).astype(int)
            f = f1_score(y_true[:, i], pred, zero_division=0)
            if f > best_f1: best_f1 = f; best_t = t
        optimal.append(best_t)
    return optimal

def find_optimal_thresholds_accuracy(y_true, y_pred_probs, rounds=4, step=0.02):
    thresholds = find_optimal_thresholds(y_true, y_pred_probs)
    best_acc = evaluate(y_pred_probs, y_true, threshold1=thresholds, threshold2=0.3)[2]
    for r in range(rounds):
        for i in range(y_true.shape[1]):
            best_t = thresholds[i]
            for t in np.arange(0.05, 0.95, step):
                trial = list(thresholds)
                trial[i] = t
                _, _, acc, _, _ = evaluate(y_pred_probs, y_true, threshold1=trial, threshold2=0.3)
                if acc > best_acc: best_acc = acc; best_t = t
            thresholds[i] = best_t
    return thresholds

def calculate_mcc(tp, tn, fp, fn):
    num = (tp * tn) - (fp * fn)
    den_sq = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return num / math.sqrt(den_sq) if den_sq > 0 else 0.0

def feature_mixup(features, labels, alpha=0.4):
    batch_size = features.size(0)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(batch_size, device=features.device)
    mixed_features = lam * features + (1 - lam) * features[index]
    mixed_labels = lam * labels + (1 - lam) * labels[index]
    return mixed_features, mixed_labels

def rdrop_kl_loss(logits1, logits2):
    kl1 = F.binary_cross_entropy_with_logits(logits1, torch.sigmoid(logits2).detach(), reduction='mean')
    kl2 = F.binary_cross_entropy_with_logits(logits2, torch.sigmoid(logits1).detach(), reduction='mean')
    return (kl1 + kl2) / 2

def enable_dropout(model):
    """
    Selectively enables dropout-related layers for Test-Time Augmentation (TTA)
    while keeping BatchNorm layers strictly frozen in eval() mode.
    """
    for module in model.modules():
        # 1. Catch all standard PyTorch dropouts (Dropout, Dropout1d, Dropout2d, etc.)
        # This will also naturally catch the Dropouts inside the Hugging Face ESM models
        if isinstance(module, nn.modules.dropout._DropoutNd):
            module.train()
            
        # 2. Catch complex modules that handle their own internal dropout
        elif isinstance(module, (
            nn.MultiheadAttention, 
            nn.TransformerDecoderLayer, 
            nn.TransformerDecoder, 
            nn.GRU
        )):
            module.train()
            
        # 3. Fallback catch just in case Hugging Face updates their backend 
        # to use a custom dropout class name that doesn't inherit from _DropoutNd
        elif "Dropout" in module.__class__.__name__:
            module.train()


# --- SWA ---
swa_start_epoch = 30
swa_model = None
swa_n = 0

def update_swa(model_state, swa_state, n):
    if swa_state is None:
        return {k: v.clone() for k, v in model_state.items()}, 1
    for k in swa_state:
        swa_state[k] = (swa_state[k] * n + model_state[k]) / (n + 1)
    return swa_state, n + 1

# --- MAIN TRAINING LOOP (v19: Triple CrossAttn + Task Query Decoder + ASL) ---
history_log = []
history_log_path = "MCMFPP_train_history.json"
best_accuracy = 0.0
scaler = torch.amp.GradScaler('cuda')

mixup_alpha = 0.4
rdrop_alpha = 1.0
tta_passes = 5



for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0
    spec_y_train, spec_pred_train = [], []

    for data in train_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        labels = data['labels'].to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            # Two forward passes for R-Drop
            _, features1 = model.get_features(input_ids, attention_mask, mask_tokens=True, mask_prob=0.15)
            _, features2 = model.get_features(input_ids, attention_mask, mask_tokens=True, mask_prob=0.15)

            logits1_clean = model.classify_features(features1)
            logits2_clean = model.classify_features(features2)

            # R-Drop consistency
            rdrop_loss = rdrop_kl_loss(logits1_clean, logits2_clean)

            # Mixup on first pass
            mixed_features, mixed_labels = feature_mixup(features1, labels, alpha=mixup_alpha)
            logit_mixed =  model.classify_features(mixed_features)

            # ASL loss
            asl_loss1 = criterion(logit_mixed, mixed_labels)
            asl_loss2 = criterion(logits2_clean, labels)

            

            # Orthogonality loss on multi-head attention pools
            ortho = model.ortho_loss()

            loss = (asl_loss1 + asl_loss2) / 2 + rdrop_alpha * rdrop_loss + 0.1 * ortho

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        total_train_loss += loss.item() * input_ids.size(0)


        spec_y_train.append(labels.cpu().numpy())
        spec_pred_train.append(torch.sigmoid(logits1_clean).detach().cpu().numpy())


    y_true_train = np.concatenate(spec_y_train)
    y_pred_train = np.concatenate(spec_pred_train)
    func_names = [index_endpoint[i] for i in FUNC_INDICES]

    train_thresh = [0.6]*21 #find_optimal_thresholds_accuracy(y_true_train, y_pred_train, rounds=4, step=0.01)

    # SWA
    if epoch >= swa_start_epoch:
        swa_model, swa_n = update_swa(ema.shadow.state_dict(), swa_model, swa_n)



    avg_train_loss = total_train_loss / len(train_loader.dataset)


    lr_now = optimizer.param_groups[0]['lr']
    swa_tag = f" [SWA n={swa_n}]" if epoch >= swa_start_epoch else ""
    print(f"Ep {epoch:02d}/{epochs} | LR: {lr_now:.1e} | Train: {avg_train_loss:.4f}")
    
    print("-" * 70)

    epoch_data = {
        "epoch": epoch, 
        "train_loss": avg_train_loss, 
       
        "thresholds_train": train_thresh

    }
    history_log.append(epoch_data)
    with open(history_log_path, "w") as f:
        json.dump(history_log, f, indent=2)


print(f"\nTraining Complete! Best Accuracy: {best_accuracy:.4f}")

Ep 01/40 | LR: 3.3e-06 | Train: 0.6690
----------------------------------------------------------------------
Ep 04/40 | LR: 1.0e-05 | Train: 0.6129
----------------------------------------------------------------------
Ep 05/40 | LR: 9.9e-06 | Train: 0.6035
----------------------------------------------------------------------
Ep 06/40 | LR: 9.8e-06 | Train: 0.5943
----------------------------------------------------------------------
Ep 07/40 | LR: 9.7e-06 | Train: 0.5853
----------------------------------------------------------------------
Ep 08/40 | LR: 9.6e-06 | Train: 0.5782
----------------------------------------------------------------------
Ep 09/40 | LR: 9.4e-06 | Train: 0.5699
----------------------------------------------------------------------
Ep 10/40 | LR: 9.1e-06 | Train: 0.5634
----------------------------------------------------------------------
Ep 11/40 | LR: 8.9e-06 | Train: 0.5582
----------------------------------------------------------------------
Ep 12/40 |

In [21]:
# =====================================================================
# PAPER-STYLE EVALUATION (Table 4 protocol from Zhao et al. 2025)
# =====================================================================
import copy

# 1. Load the best saved model
# Use EMA shadow for evaluation
eval_model = copy.deepcopy(ema.shadow)
eval_model.eval()

# OR if SWA is available and epoch >= 30:
if swa_model is not None:
    eval_model = copy.deepcopy(ema.shadow)
    eval_model.load_state_dict(swa_model)
    eval_model.eval()


# 2. Run inference on full test set (eval mode, NO TTA)
all_preds = []
all_y = []

with torch.no_grad():
    for data in test_loader:
        input_ids = data['input_ids'].to(DEVICE)
        attention_mask = data['attention_mask'].to(DEVICE)
        labels = data['labels'].to(DEVICE)
        all_y.append(labels.cpu().numpy())

        with torch.amp.autocast('cuda'):
            logits = eval_model(input_ids, attention_mask)
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())

y_true_full = np.concatenate(all_y)
y_pred_full = np.concatenate(all_preds)

# 3. Construct 5 random 80% test subsets and evaluate each
n_full = len(y_true_full)
n_samp = int(n_full * 0.8)
n_subsets = 5

results = {'precision': [], 'coverage': [], 'accuracy': [], 'absolute_true': [], 'absolute_false': []}

print("=" * 75)
print(f"PAPER-STYLE EVALUATION: {n_subsets} random 80% test subsets (no TTA)")
print(f"Test set: {n_full} samples, subset size: {n_samp}")
print("=" * 75)

for trial in range(n_subsets):
     
    sidx = np.random.choice(n_full, size=n_samp, replace=False)
    y_true_s = y_true_full[sidx]
    y_pred_s = y_pred_full[sidx]

    thresh = [0.6]*21 
    aim, cov, acc, abst, absf = evaluate(y_pred_s, y_true_s, threshold1=thresh, threshold2=0.3)

    results['precision'].append(aim)
    results['coverage'].append(cov)
    results['accuracy'].append(acc)
    results['absolute_true'].append(abst)
    results['absolute_false'].append(absf)

    print(f"  Subset {trial+1}: acc={acc:.3f} | prec={aim:.3f} | cov={cov:.3f}")

# 4. Print results
print("\n" + "=" * 75)
print("Table 4 format: Five Test Subset Results (Mean +/- sd)")
print("=" * 75)
print(f"{'metric':<20s} {'mean':>8s}   {'± sd':>8s}")
print("-" * 40)
for metric in ['precision', 'coverage', 'accuracy', 'absolute_true', 'absolute_false']:
    vals = np.array(results[metric])
    print(f"{metric:<20s} {np.mean(vals):>8.3f}   ± {np.std(vals):>5.3f}")

# 5. Comparison with paper's MCMFPP
print("\n" + "=" * 75)
print("Comparison with MCMFPP paper (Zhao et al. 2025)")
print("=" * 75)
print(f"{'metric':<20s} {'MCMFPP (paper)':>16s}   {'Ours':>16s}   {'diff':>8s}")
print("-" * 65)
paper = {'precision': 0.775, 'coverage': 0.741, 'accuracy': 0.723, 'absolute_true': 0.663, 'absolute_false': 0.030}
for metric in ['precision', 'coverage', 'accuracy', 'absolute_true', 'absolute_false']:
    ours = np.mean(results[metric])
    theirs = paper[metric]
    diff = ours - theirs
    arrow = "↑" if (diff > 0 and metric != 'absolute_false') or (diff < 0 and metric == 'absolute_false') else "↓"
    print(f"{metric:<20s} {theirs:>16.3f}   {ours:>16.3f}   {diff:>+7.3f} {arrow}")

del eval_model
torch.cuda.empty_cache()

PAPER-STYLE EVALUATION: 5 random 80% test subsets (no TTA)
Test set: 1975 samples, subset size: 1580
  Subset 1: acc=0.710 | prec=0.769 | cov=0.721
  Subset 2: acc=0.716 | prec=0.771 | cov=0.729
  Subset 3: acc=0.702 | prec=0.758 | cov=0.714
  Subset 4: acc=0.716 | prec=0.770 | cov=0.727
  Subset 5: acc=0.708 | prec=0.764 | cov=0.719

Table 4 format: Five Test Subset Results (Mean +/- sd)
metric                   mean       ± sd
----------------------------------------
precision               0.766   ± 0.005
coverage                0.722   ± 0.005
accuracy                0.711   ± 0.005
absolute_true           0.656   ± 0.005
absolute_false          0.032   ± 0.001

Comparison with MCMFPP paper (Zhao et al. 2025)
metric                 MCMFPP (paper)               Ours       diff
-----------------------------------------------------------------
precision                       0.775              0.766    -0.009 ↓
coverage                        0.741              0.722    -0.019 ↓
accur